In [1]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, confusion_matrix

def churn_logistic_model(df, feature_cols, target_col):
    # ── Input validation ──────────────────────────────────
    assert len(feature_cols) > 0,                          "feature_cols must not be empty"
    assert target_col in df.columns,                       f"{target_col} not in DataFrame"
    assert all(c in df.columns for c in feature_cols),     "some feature_cols missing"
    assert df[target_col].nunique() == 2,                  "target must be binary (0/1)"

    # ── Step 1: Standardize features ─────────────────────
    scaler = StandardScaler()
    X = scaler.fit_transform(df[feature_cols])
    y = df[target_col].values

    # ── Step 2: Fit logistic regression ──────────────────
    logreg = LogisticRegression(max_iter=1000)
    logreg.fit(X, y)

    # ── Step 3: Extract coefficients + odds ratios ───────
    coefficients = logreg.coef_[0]
    odds_ratios  = np.exp(coefficients)

    # ── Step 4: Build ranking DataFrame ──────────────────
    feature_ranking = pd.DataFrame({
        'feature'    : feature_cols,
        'coefficient': coefficients,
        'odds_ratio' : odds_ratios
    }).sort_values('coefficient', key=abs, ascending=False)\
      .reset_index(drop=True)

    top_churn_driver = feature_ranking.iloc[0]['feature']

    # ── Step 5: AUC ───────────────────────────────────────
    auc = roc_auc_score(y, logreg.predict_proba(X)[:, 1])

    # ── Step 6: Confusion matrix metrics ─────────────────
    y_pred = logreg.predict(X)
    tn, fp, fn, tp = confusion_matrix(y, y_pred).ravel()
    #                                     ┌──────────────────────────────────┐
    #  confusion_matrix returns:          │  actual 0   │  actual 1          │
    #    [[TN, FP],                       ├─────────────┼────────────────────┤
    #     [FN, TP]]                       │  pred 0: TN │  pred 0: FN        │
    #  .ravel() flattens to 1D →          │  pred 1: FP │  pred 1: TP        │
    #    (TN, FP, FN, TP)                 └──────────────────────────────────┘

    precision   = tp / (tp + fp)   # of predicted churns, how many actually churned
    recall      = tp / (tp + fn)   # of actual churns, how many did we catch
    specificity = tn / (tn + fp)   # of actual non-churns, how many did we correctly ignore
    f1          = 2 * (precision * recall) / (precision + recall)
    accuracy    = (tp + tn) / (tp + tn + fp + fn)
    fpr         = fp / (fp + tn)   # false positive rate = 1 - specificity
    fnr         = fn / (fn + tp)   # false negative rate = 1 - recall

    # ── Output validation ─────────────────────────────────
    assert 0 <= auc <= 1
    assert 0 <= precision <= 1
    assert 0 <= recall <= 1
    assert 0 <= f1 <= 1
    assert 0 <= accuracy <= 1
    assert len(feature_ranking) == len(feature_cols)
    assert abs(fpr + specificity - 1) < 1e-9,  "fpr + specificity must = 1"
    assert abs(fnr + recall - 1) < 1e-9,       "fnr + recall must = 1"

    return {
        'feature_ranking' : feature_ranking,
        'top_churn_driver': top_churn_driver,
        'auc'             : round(auc, 4),
        'n_obs'           : len(df),
        'churn_rate'      : y.mean(),
        # ── confusion matrix ──
        'tp'              : int(tp),
        'tn'              : int(tn),
        'fp'              : int(fp),
        'fn'              : int(fn),
        # ── derived metrics ───
        'precision'       : round(precision,   4),
        'recall'          : round(recall,      4),
        'specificity'     : round(specificity, 4),
        'f1'              : round(f1,          4),
        'accuracy'        : round(accuracy,    4),
        'fpr'             : round(fpr,         4),
        'fnr'             : round(fnr,         4),
    }



In [5]:
# ── Sample data ───────────────────────────────────────────
np.random.seed(42)
n = 2000

df = pd.DataFrame({
    'sessions_day1_3'  : np.random.poisson(5, n),
    'avg_session_min'  : np.random.exponential(15, n),
    'friends_added'    : np.random.poisson(2, n),
    'levels_completed' : np.random.poisson(3, n),
    'spent_robux'      : np.random.exponential(50, n),
    'tutorial_complete': np.random.binomial(1, 0.6, n),
})

churn_score = (
    -0.8 * df['sessions_day1_3']
    -0.5 * df['tutorial_complete']
    -0.3 * df['friends_added']
    + np.random.normal(0, 1, n)
)
df['churned'] = (churn_score > churn_score.median()).astype(int)

feature_cols = ['sessions_day1_3', 'avg_session_min', 'friends_added',
                'levels_completed', 'spent_robux', 'tutorial_complete']

result = churn_logistic_model(df, feature_cols, target_col='churned')

print(f"AUC:              {result['auc']}")
print(f"N observations:   {result['n_obs']}")
print(f"Baseline churn:   {result['churn_rate']:.2%}")
print(f"Top churn driver: {result['top_churn_driver']}")
print(f"\nFeature Ranking:")
print(result['feature_ranking'].to_string(index=False))

print ("============")
for k,v in result.items():
    if k not in ['feature_ranking', 'top_churn_driver']:
        print(f"{k:20s}: {v}")



AUC:              0.9301
N observations:   2000
Baseline churn:   50.00%
Top churn driver: sessions_day1_3

Feature Ranking:
          feature  coefficient  odds_ratio
  sessions_day1_3    -3.073388    0.046264
    friends_added    -0.914682    0.400644
tutorial_complete    -0.444235    0.641315
  avg_session_min    -0.113896    0.892350
      spent_robux    -0.043234    0.957687
 levels_completed    -0.032210    0.968304
auc                 : 0.9301
n_obs               : 2000
churn_rate          : 0.5
tp                  : 854
tn                  : 821
fp                  : 179
fn                  : 146
precision           : 0.8267
recall              : 0.854
specificity         : 0.821
f1                  : 0.8401
accuracy            : 0.8375
fpr                 : 0.179
fnr                 : 0.146
